# 2. Building the scoring machine (weeks 5-7)
We train two models on **2007-2013** loans and judge them on **2014-15** loans, then compare with LendingClub's grade.

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
import numpy as np
import plotly.express as px
from lc_rates import config, data, features, finance, model
pd.options.display.float_format = "{:,.4f}".format

In [2]:
from sklearn.metrics import roc_auc_score
df = data.load_clean()
train, test = data.split_train_test(df)
print(f'train {len(train):,} loans ({train.issue_year.min()}-{train.issue_year.max()}), test {len(test):,} loans ({test.issue_year.min()}-{test.issue_year.max()})')

train 227,957 loans (2007-2013), test 445,357 loans (2014-2015)


## Step 1: the inputs
The model may only use what is known **at the moment of application**. It must never see the grade, the interest rate, or anything about repayment. Check that our feature list obeys this.

In [3]:
X_train, X_test = features.build_features(train), features.build_features(test)
y_train, y_test = train['defaulted'].to_numpy(), test['defaulted'].to_numpy()
print(features.FEATURE_COLUMNS)
banned = set(config.LC_VERDICT_COLUMNS) | set(config.OUTCOME_COLUMNS) | {'installment'}
assert not banned & set(features.FEATURE_COLUMNS)
X_train.head()

['loan_amnt', 'term_months', 'log_annual_inc', 'dti', 'fico_mid', 'emp_length_years', 'revol_util', 'log_revol_bal', 'delinq_2yrs', 'inq_last_6mths', 'pub_rec', 'open_acc', 'total_acc', 'credit_history_years', 'loan_to_income', 'purpose', 'home_ownership']


,loan_amnt,term_months,log_annual_inc,dti,fico_mid,emp_length_years,revol_util,log_revol_bal,delinq_2yrs,inq_last_6mths,pub_rec,open_acc,total_acc,credit_history_years,loan_to_income,purpose,home_ownership
0,"5,000.0000",36.0000,10.0858,27.6500,737.0000,10.0000,83.7000,9.5214,0.0000,1.0000,0.0000,3.0000,9.0000,26.9131,0.2083,credit_card,RENT
1,"2,500.0000",60.0000,10.3090,1.0000,742.0000,0.0000,9.4000,7.4313,0.0000,5.0000,0.0000,3.0000,4.0000,12.6680,0.0833,car,RENT
2,"2,400.0000",36.0000,9.4134,8.7200,737.0000,10.0000,98.5000,7.9919,0.0000,2.0000,0.0000,2.0000,10.0000,10.0808,0.1959,small_business,RENT
3,"10,000.0000",36.0000,10.8036,20.0000,692.0000,10.0000,21.0000,8.6303,0.0000,1.0000,0.0000,10.0000,37.0000,15.8303,0.2033,other,RENT
4,"3,000.0000",60.0000,11.2898,17.9400,697.0000,1.0000,53.9000,10.2322,0.0000,0.0000,0.0000,15.0000,38.0000,15.9151,0.0375,other,RENT


`installment` is a trap: it is calculated *from the interest rate*, so it would sneak LendingClub's own price into our 'independent' model.

## Model 1: logistic regression (explainable)

In [4]:
logit = model.make_logistic().fit(X_train, y_train)
p_logit = model.predict_pd(logit, X_test)
print('AUC', round(roc_auc_score(y_test, p_logit), 4))
model.logistic_coefficients(logit).head(12)

AUC 0.6554


,feature,coef,odds_ratio
0,cat__purpose_small_business,0.5886,1.8014
1,cat__home_ownership_MORTGAGE,-0.4717,0.6239
2,cat__purpose_credit_card,-0.4583,0.6324
3,cat__home_ownership_OWN,-0.4094,0.6640
4,cat__purpose_car,-0.3697,0.6909
5,cat__purpose_wedding,-0.3451,0.7082
6,num__term_months,0.3338,1.3962
7,num__fico_mid,-0.3274,0.7208
8,cat__home_ownership_RENT,-0.3001,0.7407
9,cat__purpose_major_purchase,-0.2942,0.7451


A coefficient above 0 pushes risk up; below 0 pushes it down. Numeric inputs are standardised, so coefficients are comparable.

## Model 2: LightGBM (stronger)
Training takes about a minute, so we load the one saved by `scripts/train.py`.

In [5]:
logit, lgbm, meta = model.load_models()
p_lgbm = model.predict_pd(lgbm, X_test)
print('trees used:', meta['best_iteration'])
model.probability_quality(y_test, p_lgbm)

trees used: 150


{'auc': 0.6616802662371005,
 'brier': 0.11897556861219456,
 'log_loss': 0.39356944834639424,
 'mean_predicted': 0.1299530994587398,
 'actual_default_rate': 0.14461207525647962}

## Step 3: against LendingClub
AUC asks: pick one borrower who defaulted and one who did not; how often does the score rank the defaulter as riskier? 0.5 is a coin flip.

In [6]:
scores = {
    'Logistic (mine)': p_logit, 'LightGBM (mine)': p_lgbm,
    'LendingClub grade': model.grade_to_score(test['grade']).to_numpy(),
    'LendingClub sub-grade': model.sub_grade_to_score(test['sub_grade']).to_numpy(),
    'LendingClub interest rate': test['int_rate'].to_numpy(),
}
auc = model.score_summary(y_test, scores)
px.bar(auc.sort_values('auc'), x='auc', y='score', orientation='h', range_x=[0.5, 0.7], text_auto='.3f').show()
auc

,score,auc
0,LendingClub sub-grade,0.6712
1,LendingClub interest rate,0.6675
2,LightGBM (mine),0.6617
3,LendingClub grade,0.6608
4,Logistic (mine),0.6554


LendingClub's finer sub-grade edges out my model: it sees credit-bureau details my 15 inputs do not. But is that the end of the story?

In [7]:
res = pd.read_csv(config.RESULTS_DIR / 'incremental_information.csv')
res

,inputs,cv_auc
0,LendingClub sub-grade only,0.6707
1,My LightGBM only,0.6616
2,Both together,0.6842


Using **both** beats either alone: my machine knows something LendingClub's grade does not. Where exactly do they disagree?

In [8]:
model.disagreement_table(test, p_lgbm)

,group,loans,actual_default_rate,mean_interest_rate,mean_model_pd
0,"LC says safe, model says risky",7592,0.1200,0.0804,0.1892
1,"LC says risky, model says safe",9108,0.1502,0.1514,0.0737
2,Both say safe,84940,0.0454,0.0715,0.0576
3,Both say risky,77731,0.2721,0.1652,0.2181


## Is the probability itself right? (calibration)

In [9]:
d = model.decile_table(y_test, p_lgbm)
fig = px.line(d, x='mean_predicted_pd', y='actual_default_rate', markers=True, title='Predicted vs actual default rate')
fig.add_shape(type='line', x0=0, y0=0, x1=0.4, y1=0.4, line=dict(dash='dash'))
fig.update_layout(xaxis_tickformat='.0%', yaxis_tickformat='.0%').show()
d

,decile,loans,mean_predicted_pd,actual_default_rate
0,1,44536,0.0413,0.0385
1,2,44536,0.0649,0.0701
2,3,44535,0.0809,0.0892
3,4,44536,0.0960,0.1109
4,5,44536,0.1113,0.1272
5,6,44535,0.1280,0.1488
6,7,44536,0.1471,0.1673
7,8,44535,0.1700,0.1912
8,9,44536,0.2002,0.2219
9,10,44536,0.2600,0.2811


## Why did it say that? (SHAP)

In [10]:
imp, values, Xs = model.shap_importance(lgbm, X_test, sample=5000)
px.bar(imp.head(12).sort_values('mean_abs_shap'), x='mean_abs_shap', y='feature', orientation='h').show()
imp.head(8)

/Users/danielngo/Desktop/Imp/Python Project/lendingclub-fair-rates/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/danielngo/Desktop/Imp/Python Project/lendingclub-fair-rates/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,mean_abs_shap
0,fico_mid,0.2831
1,term_months,0.1887
2,log_annual_inc,0.1537
3,inq_last_6mths,0.1347
4,loan_to_income,0.1331
5,dti,0.0911
6,purpose,0.0898
7,revol_util,0.0716


### Your turn
1. Add one more feature (for example `mort_acc` from the raw file) and see whether AUC improves. Remember to check it is known at application time.
2. What happens to AUC if you *wrongly* add `int_rate` as an input? Explain why that would be cheating.
3. The model under-predicts default overall. Why might 2014-15 differ from 2007-13?